# Configuração ambiente

In [0]:
%sql
USE CATALOG medalhao;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import timedelta


In [0]:
catalogo = "medalhao"
silver_db_name = "silver"

In [0]:
spark.sql("DROP TABLE IF EXISTS silver.ft_consumidores")
spark.sql("DROP TABLE IF EXISTS silver.ft_itens_pedidos")
spark.sql("DROP TABLE IF EXISTS silver.ft_pagamentos_pedidos")
spark.sql("DROP TABLE IF EXISTS silver.ft_avaliacoes_pedidos")
spark.sql("DROP TABLE IF EXISTS silver.ft_pedidos")
spark.sql("DROP TABLE IF EXISTS silver.ft_produtos")
spark.sql("DROP TABLE IF EXISTS silver.ft_vendedores")
spark.sql("DROP TABLE IF EXISTS silver.dm_categoria_produtos_traducao")

#Carregamento tabelas

In [0]:
ft_consumidores_bronze_df = spark.table("medalhao.bronze.ft_consumidores")
ft_pedidos_bronze_df = spark.table("medalhao.bronze.ft_pedidos")
ft_itens_pedidos_bronze_df = spark.table("medalhao.bronze.ft_itens_pedidos")
ft_pagamentos_pedidos_bronze_df = spark.table("medalhao.bronze.ft_pagamentos_pedidos")
ft_avaliacoes_pedidos_bronze_df = spark.table("medalhao.bronze.ft_avaliacoes_pedidos")
ft_produtos_bronze_df = spark.table("medalhao.bronze.ft_produtos")
ft_vendedores_bronze_df = spark.table("medalhao.bronze.ft_vendedores")
dm_categoria_produtos_bronze_traducao_df = spark.table("medalhao.bronze.dm_categoria_produtos_traducao")
dm_cotacao_dolar_bronze_df = spark.table("medalhao.bronze.dm_cotacao_dolar")

##Ft_consumidores
A coluna id_consumidor não deve conter valores duplicados, é necessário realizar essa verificação antes de
salvar a tabela na camada Silver (Não utilize a customer_unique_id)


Nomes de Estado e Cidade devem estar em Upper Case (Em letras maiúsculas)


In [0]:
ft_consumidores_bronze_df.printSchema()

In [0]:
ft_consumidores_silver_df = (
    ft_consumidores_bronze_df
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("customer_city")).alias("cidade"),
        F.upper(F.col("customer_state")).alias("estado"),
        F.col("ingestion_timestamp").alias("tempo_ingestao")
    )
    .filter(F.col("id_consumidor").isNotNull())
    .dropDuplicates(["id_consumidor"])
)


ft_consumidores_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_consumidores")
display(ft_consumidores_silver_df.limit(5))

##Ft_pedidos
Será necessário converter a coluna order_status do dataset da camada Bronze, que originalmente estavam em
inglês, para seus equivalentes em português, de forma a padronizar e facilitar a interpretação dos dados na
camada Silver

Será necessário criar novas colunas com informações derivadas, a fim de analisar as análises das áreas de
negócio

1. tempo_entrega_dias → diferença em dias entre a data de entrega (pedido_entregue_timestamp) e a data de
compra (pedido_compra_timestamp);

2. tempo_entrega_estimado_dias → diferença em dias entre a data estimada de entrega
(pedido_estimativa_entrega_timestamp) e a data de compra (pedido_compra_timestamp);

3. diferenca_entrega_dias → diferença entre o tempo real e o tempo estimado de entrega;

4. entrega_no_prazo → indicador textual que deve conter:

"Sim" → quando a entrega ocorreu no prazo (diferença ≤ 0);
"Não" → quando ocorreu fora do prazo;
"Não Entregue" → quando o pedido ainda não foi entregue.


In [0]:
# de -> para da coluna order_status
mapeamento_pedidos = {
    "delivered": "entregue",
    "invoiced": "faturado",
    "shipped": "enviado",
    "processing": "em processamento",
    "unavailable": "indisponível",
    "canceled": "cancelado",
    "created": "criado",
    "approved": "aprovado"
}

In [0]:
ft_pedidos_silver_df = (
    ft_pedidos_bronze_df
    #traduções
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),
        F.col("order_status").alias("status"),
        F.col("order_purchase_timestamp").alias("pedido_compra_timestamp"),
        F.col("order_approved_at").alias("pedido_aprovado_timestamp"),
        F.col("order_delivered_carrier_date").alias("pedido_carregado_timestamp"),
        F.col("order_delivered_customer_date").alias("pedido_entregue_timestamp"),
        F.col("order_estimated_delivery_date").alias("pedido_estimativa_entrega_timestamp"),
        F.col("ingestion_timestamp").alias("tempo_ingestao")
    )
    #coluna 
    .withColumn(
        "status",
        F.create_map([F.lit(x) for x in sum(mapeamento_pedidos.items(), ())])[F.col("status")]
    )
    .withColumn("tempo_entrega_dias", F.datediff(F.col("pedido_entregue_timestamp"), F.col("pedido_compra_timestamp")))
    .withColumn("tempo_entrega_estimado_dias", F.datediff(F.col("pedido_estimativa_entrega_timestamp"), F.col("pedido_compra_timestamp")))
    .withColumn("diferenca_entrega_dias", F.datediff(F.col("pedido_entregue_timestamp"), F.col("pedido_estimativa_entrega_timestamp")))
    .withColumn("entrega_no_prazo", F.when(F.col("diferenca_entrega_dias") <= 0, "Sim")
                                    .when(F.col("status") != "entregue", "Não Entregue")
                                    .otherwise("Não"))



)
ft_pedidos_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_pedidos")
display(ft_pedidos_silver_df.limit(5))

##Ft_intes_pedidos


In [0]:
ft_itens_pedidos_silver_df = (
    ft_itens_pedidos_bronze_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("order_item_id").alias("id_item"),
        F.col("product_id").alias("id_produto"),
        F.col("seller_id").alias("id_vendedor"),
        F.col("price").alias("preco_BRL").cast("decimal(10,2)"),
        F.col("freight_value").alias("preco_frete").cast("decimal(10,2)"),
        F.col("ingestion_timestamp").alias("tempo_ingestao")
    )
)
ft_itens_pedidos_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_itens_pedidos")
display(ft_itens_pedidos_silver_df.limit(5))

##Ft_pagamentos

Será necessário converter a coluna order_status do dataset da camada Bronze, que originalmente estavam em
inglês, para seus equivalentes em português, de forma a padronizar e facilitar a interpretação dos dados na
camada Silver

In [0]:
# de -> para da coluna order_status
mapeamento_pagamentos = {
    "credit_card": "Cartão de Crédito",
    "boleto": "Boleto",
    "voucher": "Voucher",
    "debit_card": "Cartão de Débito"
}

In [0]:
ft_pagamentos_pedidos_silver_df = (
    ft_pagamentos_pedidos_bronze_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("codigo_pagamento"),
        F.col("payment_type").alias("forma_pagamento"),
        F.col("payment_installments").alias("parcelas"),
        F.col("payment_value").alias("valor_pagamento"),
        F.col("ingestion_timestamp").alias("tempo_ingestao")
    )
    .withColumn(
        "forma_pagamento",
        F.coalesce( 
            F.create_map([F.lit(x) for x in sum(mapeamento_pagamentos.items(), ())])[F.col("forma_pagamento")],
            F.lit("Outro")  
        )
    )
)
    
ft_pagamentos_pedidos_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_pagamentos_pedidos")
display(ft_pagamentos_pedidos_silver_df.limit(5))
     

##Ft_avaliacoes_pedidos

Remover registros em ft_avaliacoes_pedidos que tenham id_pedido inválido ou datas incorretas (ex.: data nula, formato
inconsistente, data futura fora do escopo).


Documente as regras exatas de validação (o que é considerado "ID incorreto" e "data preenchida errada") no
notebook e registre o número de linhas removidas.

Regras:
1. Validação do id_pedido

Um id_pedido é considerado inválido e será removido se:
O valor for NULO.

O ID não tiver exatamente 32 caracteres (padrão UUID do Olist).

O ID contiver caracteres que não sejam hexadecimais (permitido apenas a-f e 0-9).


2. Validação das Datas

Um registro é considerado inválido e será removido se:

data_comentario ou data_resposta forem NULAS ou tiverem um formato de data/hora inválido (não conversível para timestamp).

A data_resposta for anterior à data_comentario (a resposta não pode vir antes do comentário).

Qualquer uma das datas (data_comentario ou data_resposta) estiver no futuro (data maior que o momento atual do processamento).


In [0]:

ft_avaliacoes_pedidos_silver_df = (
    ft_avaliacoes_pedidos_bronze_df
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        F.col("review_score").alias("avaliacao").cast("integer"),
        F.col("review_comment_title").alias("titulo_comentario"),
        F.col("review_comment_message").alias("comentario"),
        F.col("review_creation_date").alias("data_comentario"),
        F.col("review_answer_timestamp").alias("data_resposta"),
        F.col("ingestion_timestamp").alias("tempo_ingestao")
    )
    # Filtra apenas id_pedido válidos (conforme suas regras)
    .filter(
        (F.col("id_pedido").isNotNull()) &
        (~F.col("id_pedido").rlike(r"\s")) &
        (F.col("id_pedido").rlike(r"^[a-z0-9]+$")) &
        (~F.col("id_pedido").rlike(r"^\d{4}-\d{2}-\d{2}$"))
    )
    # Converte as colunas de data para timestamp, quando não é possível converter, substitui por nulo
    .withColumn("data_comentario", F.try_to_timestamp("data_comentario"))
    .withColumn("data_resposta", F.try_to_timestamp("data_resposta"))
    # Filtra datas
    .filter(
        (F.col("data_comentario").isNotNull()) &
        (F.col("data_resposta").isNotNull()) &
        (F.col("data_comentario") < F.col("data_resposta")) &
        (F.col("data_comentario") < F.current_timestamp()) &
        (F.col("data_resposta") < F.current_timestamp())
    )
)

ft_avaliacoes_pedidos_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_avaliacoes_pedidos")
display(ft_avaliacoes_pedidos_silver_df.limit(10))

##Ft_produtos

In [0]:
ft_produtos_silver_df = (
    ft_produtos_bronze_df
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_category_name").alias("categoria_produto"),
        F.col("product_weight_g").alias("peso_produto_gramas"),
        F.col("product_length_cm").alias("comprimento_centimetros"),
        F.col("product_height_cm").alias("altura_centimetros"),
        F.col("product_width_cm").alias("largura_centimetros"),
        F.col("ingestion_timestamp").alias("tempo_ingestao")
    )
)

ft_produtos_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_produtos")
display(ft_produtos_silver_df.limit(5))

##Ft_vendedores

In [0]:
ft_vendedores_silver_df = (
    ft_vendedores_bronze_df
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("seller_city")).alias("cidade"),
        F.upper(F.col("seller_state")).alias("estado"),
        F.col("ingestion_timestamp").alias("tempo_ingestao")
    )
)
    
ft_vendedores_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_vendedores")
display(ft_vendedores_silver_df.limit(5))

##dm_categoria_produtos_traducao

In [0]:
dm_categoria_produtos_silver_traducao_df = (
    dm_categoria_produtos_bronze_traducao_df
    .select(
        F.col("product_category_name").alias("nome_produto_pt"),
        F.col("product_category_name_english").alias("nome_produto_en"),
        F.col("ingestion_timestamp").alias("tempo_ingestao")
    )
)

dm_categoria_produtos_silver_traducao_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.dm_categoria_produtos_traducao")
display(dm_categoria_produtos_silver_traducao_df.limit(5))

##Dm_cotacao_dolar
A API não fornece cotação para finais de semana. Substitua faltas por cotação de fechamento da sexta-feira
anterior.

Sugestão técnica: calcular a última cotação disponível usando window functions do spark.
Coluna na Bronze Coluna na Silver


Tipagem:
Converta tipos de dados para o tipo correto (strings que representam inteiros → INT , datas → TIMESTAMP / DATE ,
valores monetários → DECIMAL(12,2) ou FLOAT conforme necessidade)

In [0]:
dm_cotacao_dolar_silver_df = (
    dm_cotacao_dolar_bronze_df
    .select(
        F.col("cotacaoCompra").alias("cotacao_dolar"),
        F.col("ingestion_timestamp").alias("tempo_ingestao"),
        F.to_timestamp(F.col("dataHoraCotacao"), "yyyy-MM-dd HH:mm:ss.SSS").alias("data")
    )
)

In [0]:
primeira_compra = dm_cotacao_dolar_silver_df.select(F.min("data")).collect()[0][0].date()
ultima_compra = dm_cotacao_dolar_silver_df.select(F.max("data")).collect()[0][0].date()

datas_fim_de_semana = (
    spark.createDataFrame([(1,)], ["dummy"])  
    .select(
        F.explode(
            F.sequence(F.to_date(F.lit(primeira_compra)), F.to_date(F.lit(ultima_compra)), F.expr("interval 1 day"))
        ).alias("date_only")
    )
    .withColumn("data", F.to_timestamp("date_only"))  
    .select("data")
    .filter((F.dayofweek(F.col("data")) == 1) | (F.dayofweek(F.col("data")) == 7)) 

)

w = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, 0)

inter_datas = (
    datas_fim_de_semana
    .join(dm_cotacao_dolar_silver_df, on="data", how="outer")
)

dm_cotacao_dolar_silver_df = (
    inter_datas
    .withColumn(
        "cotacao_dolar",
        F.when(
            F.dayofweek("data").isin([1, 7]), 
            F.last("cotacao_dolar", ignorenulls=True).over(w)  
        ).otherwise(F.col("cotacao_dolar")) 
    )
)

# 6. CORREÇÃO DA ANÁLISE: Renomear 'tempo_ingestao' para 'ingestion_timestamp'
if "tempo_ingestao" in dm_cotacao_dolar_silver_df.columns:
    dm_cotacao_dolar_silver_df = dm_cotacao_dolar_silver_df \
        .withColumnRenamed("tempo_ingestao", "ingestion_timestamp")

if "ingestion_timestamp" in dm_cotacao_dolar_silver_df.columns:
    dm_cotacao_dolar_silver_df = dm_cotacao_dolar_silver_df \
        .withColumn("ingestion_timestamp", 
            F.when(
                F.col("ingestion_timestamp").isNull(), 
                F.current_timestamp()
            ).otherwise(F.col("ingestion_timestamp"))
        )

dm_cotacao_dolar_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.dm_cotacao_dolar")
display(dm_cotacao_dolar_silver_df.limit(10))

Validações:

Após o carregamento das tabelas da camada Silver ( ft_pedidos , ft_consumidores e ft_itens_pedidos ), realize uma
verificação de integridade referencial entre os dados, garantindo que:

1. Todos os pedidos possuam um consumidor válido (ou seja, não existam pedidos órfãos sem
correspondência na tabela de consumidores).

2. Todos os itens de pedidos estejam associados a um pedido existente (não existam itens órfãos sem
pedido correspondente).

Ao final de cada uma das verificações, indique a quantidade de Pedidos e Itens órfãos. Caso eles existam,
retire esses registros das tabelas
Dica: Utilize joins do tipo left_anti para identificar registros órfãos


In [0]:
pedidos_orfaos_df = ft_pedidos_silver_df.join(
    ft_consumidores_silver_df,
    on="id_consumidor",
    how="left_anti"
)

toal_pedidos_orfaos = pedidos_orfaos_df.count()

print(f"Quantidade de pedidos órfãos: {toal_pedidos_orfaos}")

In [0]:
itens_orfaos_df = ft_itens_pedidos_silver_df.join(
    ft_pedidos_silver_df,
    on="id_pedido",
    how="left_anti"
)

total_itens_orfaos = itens_orfaos_df.count()

print(f"Quantidade de itens órfãos: {total_itens_orfaos}")


Por fim, será necessário a criação da tabela silver.ft_pedido_total

Junte as fontes necessárias: bronze.ft_pedidos com bronze.ft_consumidores , bronze.ft_pagamentos_pedidos e
bronze.dm_cotacao_dolar .

A tabela final deve conter as colunas:

id_pedido

id_consumidor

status

valor_total_pago_brl (soma dos pagamentos em BRL)

valor_total_pago_usd (soma dos pagamentos convertidos para USD usando cotação da data do pedido)

data_pedido (data do pedido)


In [0]:
juncao_pedidos_consumidores_df = (
    ft_pedidos_bronze_df
    .join(
        ft_consumidores_bronze_df,
        on="customer_id",
        how="inner"
    )
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),
        F.col("order_status").alias("status"),
        F.to_timestamp("order_purchase_timestamp").alias("data_pedido")
    )
)

pedidos_consumidores_pagamentos_df = (
    juncao_pedidos_consumidores_df
    .join(
        ft_pagamentos_pedidos_bronze_df,
        juncao_pedidos_consumidores_df.id_pedido == ft_pagamentos_pedidos_bronze_df.order_id,
        how="inner"
    )
    .groupBy(
        "id_pedido", "id_consumidor", "status", "data_pedido"
    )
    .agg(
        F.sum("payment_value").alias("valor_total_pago_brl").cast("Decimal(10,2)")
    )
)

pedidos_consumidores_pagamentos_df = pedidos_consumidores_pagamentos_df.withColumn(
    "data_pedido", F.to_date("data_pedido")
)
 
dm_cotacao_dolar_df = dm_cotacao_dolar_silver_df.withColumn(
    "data", F.to_date("data")
)

df_final = (
    pedidos_consumidores_pagamentos_df
    .join(
        dm_cotacao_dolar_df.select("data", "cotacao_dolar"),
        pedidos_consumidores_pagamentos_df.data_pedido == dm_cotacao_dolar_df.data,
        how="left"
    )
    .withColumn(
        "valor_total_pago_usd",
        (F.col("valor_total_pago_brl") / F.col("cotacao_dolar")).cast("Decimal(10,2)")
    )
    .select(
        "id_pedido",
        "id_consumidor",
        "status",
        "valor_total_pago_brl",
        "valor_total_pago_usd",
        "data_pedido"
    )
)
df_final.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_pedido_total")
df_final.orderBy("data_pedido").limit(10).display()
#display(df_final.limit(10))